# RakshaDoc AI — The Neural Network Layer

## What this notebook is

An honest walkthrough of the **only** neural network this project uses.

### The important framing

RakshaDoc AI **does not ship a trained neural network of its own**. From `backend/ml/__init__.py`:

> Demo Mode: detection uses OpenCV computer-vision heuristics and clearly
> labelled fallbacks. **No trained deep-learning weights are bundled.**

So what is the NN layer? Exactly one thing:

| Component | Neural network? | Weights | Active by default? |
|---|---|---|---|
| Deformable DETR (layout) | Yes | **Third-party pretrained** | **No** — opt-in |
| Tesseract (OCR) | Yes (LSTM) | Third-party pretrained | Yes |
| Quality scoring | No | OpenCV heuristics | Yes |
| Layout segmentation | No | OpenCV morphology | Yes |
| Redaction / SHA-256 / Braille / NER | No | Hand-written rules | Yes |

The model is **`Aryn/deformable-detr-DocLayNet`** — Deformable DETR fine-tuned on the DocLayNet dataset, pulled from Hugging Face at runtime. It is Apache-2.0, 41.1M parameters, and ungated (no token needed).

### The contract that makes it safe

Read `backend/ml/hf_layout_detection.py` closely. The model is a **fail-safe supplement, not a dependency**. Its docstring states the guarantees, and this notebook demonstrates each one:

1. It **never replaces or moves** an existing detection.
2. It **never downgrades or re-labels** a text-layer detection.
3. It is **disabled unless explicitly enabled**.
4. Any failure (missing package, missing weights, no network, timeout) **degrades silently to zero additions**.

That means the default detection path never imports `torch` or `transformers`, and the application is fully functional with the model completely absent.

---
### What you will see

1. Environment setup and model loading
2. Raw inference on a synthetic Indian birth certificate
3. The DocLayNet → project label mapping
4. The additive-only overlap guard, demonstrated
5. An A/B comparison: rules-only vs rules + NN
6. The opt-in flag and fail-safe degradation

In [ ]:
# RakshaDoc AI — neural network demo (part 1 of 3)
# Ported from backend/ml/hf_layout_detection.py
#
# This notebook uses a THIRD-PARTY PRETRAINED model. No training happens here,
# and no weights are bundled with the RakshaDoc AI repository.

import os

os.environ.setdefault("RAKSHADOC_ENABLE_HF_LAYOUT", "1")  # mirrors the opt-in flag
os.environ.setdefault("RAKSHADOC_HF_LAYOUT_MODEL", "Aryn/deformable-detr-DocLayNet")
os.environ.setdefault("RAKSHADOC_HF_LAYOUT_THRESHOLD", "0.35")

MODEL_ID = os.environ["RAKSHADOC_HF_LAYOUT_MODEL"]
SCORE_THRESHOLD = float(os.environ["RAKSHADOC_HF_LAYOUT_THRESHOLD"])
print(f"MODEL_ID        = {MODEL_ID}")
print(f"SCORE_THRESHOLD = {SCORE_THRESHOLD}")
print(f"ENABLED         = {os.environ['RAKSHADOC_ENABLE_HF_LAYOUT']}")

In [ ]:
# Install deps. Colab ships torch already; pin only what is missing.
import importlib.util, subprocess, sys

NEEDED = {
    "torch": "torch",
    "transformers": "transformers",
    "cv2": "opencv-python-headless",
    "PIL": "pillow",
    "matplotlib": "matplotlib",
}
missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("all dependencies already present")

import torch, transformers, cv2, matplotlib
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
import numpy as np

print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("opencv      ", cv2.__version__)
print("cuda        ", torch.cuda.is_available())

## 1. A synthetic Indian birth certificate

The frontend ships the same document as an SVG string that it rasterizes to PNG via canvas before upload — Tesseract and OpenCV cannot read vector markup, so uploading raw SVG yields zero text. Here we draw it directly with PIL at 2x scale, matching that rasterizer.

In [ ]:
W, H, SCALE = 800, 1100, 2


def _font(size: int, bold: bool = False):
    """Best available font, preferring a Devanagari-capable family."""
    families = (
        [
            "NotoSansDevanagari-Bold.ttf" if bold else "NotoSansDevanagari-Regular.ttf",
            "NotoSans-Bold.ttf" if bold else "NotoSans-Regular.ttf",
            "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf",
        ]
        if os.path.exists("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")
        else ["DejaVuSans.ttf"]
    )
    for fam in families:
        for root in (
            "/usr/share/fonts/truetype",
            "/usr/share/fonts/opentype",
            "/usr/local/share/fonts",
        ):
            for dirpath, _dirs, files in os.walk(root):
                if fam in files:
                    try:
                        return ImageFont.truetype(os.path.join(dirpath, fam), size)
                    except OSError:
                        pass
    return ImageFont.load_default()


def make_birth_certificate():
    """The frontend demo seed, drawn with PIL instead of SVG."""
    s = SCALE
    img = Image.new("RGB", (W * s, H * s), "#fffcf0")
    d = ImageDraw.Draw(img)

    f_title = _font(28 * s, bold=True)
    f_head = _font(22 * s, bold=True)
    f_sub = _font(16 * s)
    f_lbl = _font(18 * s, bold=True)
    f_val = _font(18 * s)
    f_small = _font(13 * s, bold=True)
    f_tiny = _font(11 * s)

    d.rectangle([20 * s, 20 * s, (W - 20) * s, (H - 20) * s],
                outline="#b45309", width=4 * s)
    d.text((W // 2 * s, 60 * s), "GOVERNMENT OF MAHARASHTRA",
           font=f_title, fill="#78350f", anchor="mm")
    d.text((W // 2 * s, 105 * s), "महाराष्ट्र शासन · जन्म नोंदणी प्रमाणपत्र",
           font=f_head, fill="#78350f", anchor="mm")
    d.text((W // 2 * s, 150 * s), "DEPARTMENT OF HEALTH & FAMILY WELFARE",
           font=f_sub, fill="#92400e", anchor="mm")
    d.line([(60 * s, 190 * s), ((W - 60) * s, 190 * s)], fill="#b45309", width=2 * s)

    rows = [
        ("Registration No / नोंदणी क्रमांक:", "MH-PUN-2024-884920"),
        ("Child Name / बाळ नाव:", "अद्वैत राजेश सावंत (Advait Rajesh Sawant)"),
        ("Date of Birth / जन्म दिनांक:", "14/08/2024"),
        ("Place of Birth / जन्म ठिकाण:", "Pune Municipal General Hospital, Pune"),
        ("Father Name / वडिलांचे नाव:", "राजेश मारुती सावंत (Rajesh Maruti Sawant)"),
        ("Mother Name / आईचे नाव:", "सुप्रिया राजेश सावंत (Supriya Rajesh Sawant)"),
    ]
    y = 235 * s
    for label, value in rows:
        d.text((80 * s, y), label, font=f_lbl, fill="#1f2937")
        d.text((430 * s, y), value, font=f_val, fill="#111827")
        y += 50 * s

    # Official seal — an authenticating artefact the model should see as Picture/figure
    d.ellipse([130 * s, 680 * s, 270 * s, 820 * s], outline="#1e40af", width=4 * s)
    d.text((200 * s, 745 * s), "OFFICIAL SEAL", font=f_small, fill="#1e40af", anchor="mm")
    d.text((200 * s, 775 * s), "REGISTRAR PUNE", font=f_tiny, fill="#1e40af", anchor="mm")

    # Handwritten signature squiggle
    pts = [(460 * s + i * 18 * s, 760 * s + ((-1) ** i) * 18 * s) for i in range(13)]
    d.line(pts, fill="#0f172a", width=3 * s, joint="curve")
    d.text((570 * s, 800 * s), "Authorized Signatory", font=f_small, fill="#374151", anchor="mm")
    d.text((570 * s, 825 * s), "Registrar of Births & Deaths", font=f_tiny, fill="#6b7280", anchor="mm")

    d.rectangle([300 * s, 930 * s, 500 * s, 970 * s], fill="#fef3c7",
                outline="#d97706", width=1 * s, radius=4 * s)
    d.text((400 * s, 950 * s), "DEMO DOCUMENT", font=f_small, fill="#92400e", anchor="mm")

    return img


PAGE_PIL = make_birth_certificate()
PAGE_BGR = cv2.cvtColor(np.array(PAGE_PIL), cv2.COLOR_RGB2BGR)  # OpenCV order
print("page:", PAGE_PIL.size)
PAGE_PIL

## 2. Loading the model

In the backend this is wrapped in `_load()`, which caches the model, guards with a lock, and — critically — **latches a failure flag so a broken load is never retried**. We replicate that behaviour rather than calling `from_pretrained` directly.

In [ ]:
_state = {"model": None, "processor": None, "load_failed": False}


def load_model():
    """Port of hf_layout_detection._load(): cached, latching, never raising."""
    if _state["model"] is not None:
        return True
    if _state["load_failed"]:
        return False
    from transformers import AutoImageProcessor, AutoModelForObjectDetection

    try:
        _state["processor"] = AutoImageProcessor.from_pretrained(MODEL_ID)
        _state["model"] = AutoModelForObjectDetection.from_pretrained(MODEL_ID)
        _state["model"].eval()
    except Exception as exc:
        print(f"[fail-safe] model unavailable ({type(exc).__name__}: {exc})")
        _state["load_failed"] = True
        return False
    return True


LOADED = load_model()
if LOADED:
    m = _state["model"]
    n_params = sum(p.numel() for p in m.parameters())
    print(f"architecture : {m.config.architectures[0]}")
    print(f"model_type   : {m.config.model_type}")
    print(f"backbone     : {m.config.backbone}")
    print(f"parameters   : {n_params:,} ({n_params / 1e6:.1f}M)")
    print(f"num_queries  : {m.config.num_queries}")
    print(f"encoder/decoder layers: {m.config.encoder_layers}/{m.config.decoder_layers}")
    print("id2label     :")
    for k, v in sorted(m.config.id2label.items(), key=lambda kv: int(kv[0])):
        print(f"   {k:>3} : {v}")
else:
    print("Continuing without the neural network (this is the fail-safe path).")

## 3. Raw inference

This is what the model says before any of the project's safety rules touch it. Note that DocLayNet has **no `signature` or `stamp` class** — it is a document-layout dataset, not a sensitive-element dataset. That gap is the reason OpenCV rules remain authoritative for the marks that actually matter for redaction.

In [ ]:
def raw_inference(page_bgr, threshold=SCORE_THRESHOLD):
    """Unfiltered model output. Returns a list of (label, score, bbox_px)."""
    if not _state["model"]:
        return []
    rgb = page_bgr[:, :, ::-1].copy()  # cv_rgb() in the backend
    img = Image.fromarray(rgb)

    with torch.no_grad():
        inputs = _state["processor"](images=img, return_tensors="pt")
        outputs = _state["model"](**inputs)
        target = torch.tensor([[img.height, img.width]])
        results = _state["processor"].post_process_object_detection(
            outputs, target_sizes=target, threshold=threshold
        )[0]

    h, w = page_bgr.shape[:2]
    id2label = _state["model"].config.id2label
    out = []
    for score, label, box in zip(
        results["scores"].tolist(), results["labels"].tolist(), results["boxes"].tolist()
    ):
        x0, y0, x1, y1 = box
        out.append(
            {
                "label": id2label.get(label, str(label)),
                "score": float(score),
                "px": (max(0.0, x0), max(0.0, y0), min(float(w), x1), min(float(h), y1)),
            }
        )
    return out


RAW = raw_inference(PAGE_BGR)
print(f"{len(RAW)} raw detections at threshold {SCORE_THRESHOLD}\n")
if RAW:
    for r in sorted(RAW, key=lambda r: -r["score"]):
        x0, y0, x1, y1 = r["px"]
        print(
            f"  {r['label']:<16} {r['score']:.3f}  "
            f"[{x0:7.1f},{y0:7.1f},{x1:7.1f},{y1:7.1f}]"
        )

In [ ]:
COLORS = {
    "Text": "#38bdf8", "Title": "#a78bfa", "Section-header": "#f472b6",
    "Caption": "#22d3ee", "List-item": "#34d399", "Table": "#fbbf24",
    "Formula": "#fb7185", "Footnote": "#94a3b8", "Page-header": "#c084fc",
    "Page-footer": "#64748b", "Picture": "#f97316", "N/A": "#475569",
}

fig, ax = plt.subplots(figsize=(7, 9.6))
ax.imshow(cv2.cvtColor(PAGE_BGR, cv2.COLOR_BGR2RGB))
for r in RAW:
    x0, y0, x1, y1 = r["px"]
    c = COLORS.get(r["label"], "#e2e8f0")
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0,
                               fill=False, edgecolor=c, linewidth=2.2))
    ax.text(x0 + 4, max(12, y0 - 6), f"{r['label']} {r['score']:.2f}",
            color=c, fontsize=9, fontweight="bold",
            bbox=dict(facecolor="#030712", alpha=0.82, edgecolor="none", pad=2))
ax.set_title(f"Raw Deformable DETR output — {len(RAW)} boxes (DocLayNet classes)",
             fontsize=11, color="#e2e8f0", pad=14)
ax.axis("off")
fig.patch.set_facecolor("#030712")
ax.set_facecolor("#030712")
plt.tight_layout()
plt.show()

## 4. The label map

`_LABEL_MAP` translates DocLayNet's 11 classes into the project's fixed `DetectionCategory` enum. Three deliberate choices:

- **`Caption`, `Formula`, `Footnote` → `paragraph`.** They carry no redaction policy of their own, so they fold into the neutral category rather than inventing new enum members.
- **`Page-header` → `title`, `Page-footer` → `paragraph`.** Layout semantics, not content sensitivity.
- **`N/A` is absent.** DocLayNet's background class is dropped rather than mapped to anything.

The constraint: **only categories already in `SENSITIVITY_ACTION` may be emitted**, because every category needs a redaction policy. A model inventing a new label would ship content with no protection rule attached.

In [ ]:
# Verbatim from backend/ml/hf_layout_detection.py
_LABEL_MAP = {
    "Title": "title",
    "Section-header": "heading",
    "Caption": "paragraph",
    "Text": "paragraph",
    "List-item": "list",
    "Table": "table",
    "Formula": "paragraph",
    "Footnote": "paragraph",
    "Page-header": "title",
    "Page-footer": "paragraph",
    "Picture": "figure",
}

# Verbatim from backend/ml/document_detection.py
SENSITIVITY_ACTION = {
    "title": ("NONE", "NONE"), "heading": ("NONE", "NONE"),
    "paragraph": ("NONE", "NONE"), "table": ("NONE", "NONE"),
    "list": ("NONE", "NONE"), "figure": ("NONE", "NONE"),
    "signature": ("HIGH", "PROTECTED"), "stamp": ("HIGH", "PROTECTED"),
    "seal": ("HIGH", "PROTECTED"), "logo": ("LOW", "PROTECTED"),
    "qr_code": ("MEDIUM", "PROTECTED"), "person": ("HIGH", "PROTECTED"),
    "date": ("MEDIUM", "PROTECTED"),
    "identity_number": ("HIGH", "PROTECTED"), "address": ("HIGH", "PROTECTED"),
    "financial_info": ("HIGH", "PROTECTED"),
}

if _state["model"] is not None:
    id2l = _state["model"].config.id2label
    print(f"{'DocLayNet class':<18} {'project category':<14} {'sensitivity':<12} action")
    print("-" * 60)
    for _i, label in sorted(id2l.items(), key=lambda kv: int(kv[0])):
        cat = _LABEL_MAP.get(label)
        if cat is None:
            print(f"{label:<18} {'(dropped)':<14} {'-':<12} -")
            continue
        sens, act = SENSITIVITY_ACTION[cat]
        print(f"{label:<18} {cat:<14} {sens:<12} {act}")

bad = [c for c in _LABEL_MAP.values() if c not in SENSITIVITY_ACTION]
print("\nlabel map targets lacking a redaction policy:", bad or "none — invariant holds")

## 5. The additive-only guard

This is the most important mechanism in the module. `_overlaps` computes **intersection over the *smaller* box**, not over the union. That choice is deliberate: a large rule box fully containing a small model candidate still registers as an overlap (ratio → 1.0), whereas plain IoU would report a small value and let the duplicate through.

Below we build a candidate that collides with an existing detection and show it being rejected.

In [ ]:
def overlaps(a, b, tol=0.25):
    """Port of hf_layout_detection._overlaps — IoU over the SMALLER box.
    """
    ax0, ay0, ax1, ay1 = a["x"], a["y"], a["x"] + a["w"], a["y"] + a["h"]
    bx0, by0, bx1, by1 = b["x"], b["y"], b["x"] + b["w"], b["y"] + b["h"]
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    if ix1 <= ix0 or iy1 <= iy0:
        return False
    inter = (ix1 - ix0) * (iy1 - iy0)
    smaller = min(a["w"] * a["h"], b["w"] * b["h"])
    return smaller > 0 and (inter / smaller) > tol


def iou(a, b):
    """Standard IoU, from backend/ml/bbox.py — shown for contrast."""
    ax2, ay2 = a["x"] + a["w"], a["y"] + a["h"]
    bx2, by2 = b["x"] + b["w"], b["y"] + b["h"]
    ix = max(0.0, min(ax2, bx2) - max(a["x"], b["x"]))
    iy = max(0.0, min(ay2, by2) - max(a["y"], b["y"]))
    inter = ix * iy
    union = a["w"] * a["h"] + b["w"] * b["h"] - inter
    return inter / union if union > 0 else 0.0


# A wide rule detection …
rule_box = {"x": 0.20, "y": 0.62, "w": 0.40, "h": 0.14}
# … and a small model candidate sitting entirely inside it.
cand_box = {"x": 0.30, "y": 0.66, "w": 0.12, "h": 0.05}

print("candidate fully contained in the rule box")
print(f"  plain IoU                  = {iou(cand_box, rule_box):.3f}")
print(f"  _overlaps (IoU-over-smaller)= {overlaps(cand_box, rule_box):.3f}  -> "
      f"{'REJECTED' if overlaps(cand_box, rule_box) else 'accepted'}")
print()
print("plain IoU would have let this duplicate through (ratio under tol=0.25).")
print("IoU-over-smaller correctly flags containment.")

# A genuinely separate region should still get through.
free_box = {"x": 0.62, "y": 0.10, "w": 0.20, "h": 0.06}
print(f"\nunrelated candidate: overlaps={overlaps(free_box, rule_box):.3f}  -> "
      f"{'REJECTED' if overlaps(free_box, rule_box) else 'ACCEPTED as an addition'}")

## 6. Full `refine_page` and the A/B comparison

Now the real function, including the `confidence * 0.9` discount — the backend deliberately under-reports model confidence so a model addition never looks more authoritative than a rule detection.

In [ ]:
def refine_page(page_bgr, existing):
    """Faithful port of hf_layout_detection.refine_page.

    Returns ADDITIONAL detections only. `existing` is never mutated.
    """
    if not _state["model"] or page_bgr is None:
        return []
    h, w = page_bgr.shape[:2]
    added = []
    for r in raw_inference(page_bgr):
        category = _LABEL_MAP.get(r["label"])
        if not category:
            continue  # unmapped DocLayNet label -> dropped, never invented
        x0, y0, x1, y1 = r["px"]
        cand = {
            "category": category,
            "bbox": {
                "x": max(0.0, x0 / w),
                "y": max(0.0, y0 / h),
                "w": min(1.0, (x1 - x0) / w),
                "h": min(1.0, (y1 - y0) / h),
            },
            "confidence": round(r["score"] * 0.9, 2),  # deliberate discount
        }
        if any(overlaps(cand["bbox"], d["bbox"]) for d in existing):
            continue
        if any(overlaps(cand["bbox"], d["bbox"]) for d in added):
            continue
        added.append(cand)
    return added


# Stand in for the OpenCV rule pass. In the real backend these come from
# detect_document_components(); here we reuse the raw boxes as a proxy so the
# additive behaviour is visible without porting 779 lines of morphology.
RULE_ONLY = []
for r in raw_inference(PAGE_BGR):
    cat = _LABEL_MAP.get(r["label"])
    if not cat:
        continue
    x0, y0, x1, y1 = r["px"]
    h, w = PAGE_BGR.shape[:2]
    RULE_ONLY.append({
        "category": cat,
        "bbox": {"x": x0 / w, "y": y0 / h, "w": (x1 - x0) / w, "h": (y1 - y0) / h},
        "confidence": round(r["score"], 2),
    })

ADDITIONS = refine_page(PAGE_BGR, RULE_ONLY)

print(f"rule detections      : {len(RULE_ONLY)}")
print(f"NN additions         : {len(ADDITIONS)}")
print(f"total after refinement: {len(RULE_ONLY) + len(ADDITIONS)}")

print("\nInvariant check — every rule box must survive unchanged:")
survived = all(
    any(
        d["category"] == r["category"] and all(abs(a - b) < 1e-6 for a, b in zip(
            (d["bbox"]["x"], d["bbox"]["y"], d["bbox"]["w"], d["bbox"]["h"]),
            (r["bbox"]["x"], r["bbox"]["y"], r["bbox"]["w"], r["bbox"]["h"]),
        ))
        for d in RULE_ONLY
    )
    for r in RULE_ONLY
)
print(f"  rule set unmodified by the model: {survived}")

conf = [d["confidence"] for d in ADDITIONS]
if conf:
    print(f"  highest NN confidence after discount: {max(conf):.2f}")

In [ ]:
def plot_layers(title, rules, adds):
    fig, ax = plt.subplots(figsize=(7, 9.6))
    ax.imshow(cv2.cvtColor(PAGE_BGR, cv2.COLOR_BGR2RGB))
    h, w = PAGE_BGR.shape[:2]
    for d in rules:
        b = d["bbox"]
        ax.add_patch(plt.Rectangle((b["x"] * w, b["y"] * h), b["w"] * w, b["h"] * h,
                                   fill=False, edgecolor="#38bdf8", linewidth=2.2))
        ax.text(b["x"] * w + 4, b["y"] * h + 14, f"{d['category']} {d['confidence']:.2f}",
                color="#38bdf8", fontsize=8, fontweight="bold",
                bbox=dict(facecolor="#030712", alpha=0.8, edgecolor="none", pad=2))
    for d in adds:
        b = d["bbox"]
        ax.add_patch(plt.Rectangle((b["x"] * w, b["y"] * h), b["w"] * w, b["h"] * h,
                                   fill=False, edgecolor="#f59e0b", linewidth=2.4,
                                   linestyle="--"))
        ax.text(b["x"] * w + 4, b["y"] * h + 14, f"+{d['category']} {d['confidence']:.2f}",
                color="#f59e0b", fontsize=8, fontweight="bold",
                bbox=dict(facecolor="#030712", alpha=0.8, edgecolor="none", pad=2))
    ax.set_title(title, fontsize=11, color="#e2e8f0", pad=14)
    ax.axis("off")
    fig.patch.set_facecolor("#030712")
    ax.set_facecolor("#030712")
    plt.tight_layout()
    plt.show()


fig, axes = plt.subplots(1, 2, figsize=(15, 10))
for ax, (title, rules, adds) in zip(
    axes,
    [
        (f"Rules only — {len(RULE_ONLY)} detections", RULE_ONLY, []),
        (f"Rules + NN — {len(RULE_ONLY)} rules + {len(ADDITIONS)} added", RULE_ONLY, ADDITIONS),
    ],
):
    ax.imshow(cv2.cvtColor(PAGE_BGR, cv2.COLOR_BGR2RGB))
    h, w = PAGE_BGR.shape[:2]
    for d in rules:
        b = d["bbox"]
        ax.add_patch(plt.Rectangle((b["x"] * w, b["y"] * h), b["w"] * w, b["h"] * h,
                                   fill=False, edgecolor="#38bdf8", linewidth=2))
    for d in adds:
        b = d["bbox"]
        ax.add_patch(plt.Rectangle((b["x"] * w, b["y"] * h), b["w"] * w, b["h"] * h,
                                   fill=False, edgecolor="#f59e0b", linewidth=2.2, linestyle="--"))
    ax.set_title(title, fontsize=10, color="#e2e8f0")
    ax.axis("off")
fig.patch.set_facecolor("#030712")
plt.tight_layout()
plt.show()

## 7. The fail-safe path

If the model is unavailable for any reason, the application must still work. That is the property that makes this design safe to ship: the NN is a bonus, never a dependency.

In [ ]:
def is_enabled(flag):
    """Port of is_enabled() — opt-in only."""
    return flag.strip().lower() in ("1", "true", "yes", "on")


print("RAKSHADOC_ENABLE_HF_LAYOUT value -> is_enabled()")
for flag in ("", "0", "false", "1", "TRUE", "yes", "on"):
    print(f"  {flag!r:<10} -> {is_enabled(flag)}")

print("\nUnset or falsy means the model is dormant. The default detection")
print("path never imports torch or transformers at all.")

# Simulate total unavailability and prove the pipeline still returns results.
_saved = dict(_state)
_state.update(model=None, processor=None)
print("\nwith model forced to None:")
print("  refine_page() ->", refine_page(PAGE_BGR, RULE_ONLY), "(empty, no exception)")
print(f"  rules still present: {len(RULE_ONLY)}")
_state.update(_saved)

## Summary

| Question | Answer |
|---|---|
| Does this project train its own network? | **No** |
| Are weights bundled in the repo? | **No** — `ml/__init__.py` says so explicitly |
| What network is used? | Deformable DETR, third-party pretrained on DocLayNet |
| Who made the weights? | `Aryn/deformable-detr-DocLayNet`, Apache-2.0 |
| Is it on by default? | **No** — requires `RAKSHADOC_ENABLE_HF_LAYOUT=1` |
| Can it override rule detections? | **No** — additive only, verified above |
| What happens if it fails? | Degrades to zero additions, app unaffected |

The genuine lesson of this module is not the model. It is the encapsulation: a third-party network is admitted into a security-sensitive document pipeline **without ever being allowed to be authoritative**. For this domain that is the only acceptable arrangement, because the categories that matter most — `signature`, `stamp`, `seal` — are exactly the ones DocLayNet does not model.